# 🧹 Google Maps Reviews Cleaning

Nettoyage complet du dataset Google Maps Scraper

In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


In [3]:
# Load Google Maps data
file_path = r'C:\Users\Imadeddine\Downloads\dataset_Google-Maps-Reviews-Scraper_2026-02-27_16-18-31-987.csv'

df = pd.read_csv(file_path, sep=';', encoding='utf-8')

print(f"📊 Original Shape: {df.shape}")
print(f"\n📋 Columns: {df.columns.tolist()[:10]}... (showing first 10)")
print(f"\n🔍 Data Info:")
print(df.head(3))

📊 Original Shape: (1615, 77)

📋 Columns: ['fid', 'originalLanguage', 'reviewerId', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']... (showing first 10)

🔍 Data Info:
     fid originalLanguage   reviewerId Unnamed: 3 Unnamed: 4 Unnamed: 5  \
0  Blida         علي زبير  استقبال جيد        NaN        NaN        NaN   
1  Blida    LAMINE KEBBAB          3.5        NaN        NaN        NaN   
2  Blida   Kamel Mederbel          3.5        NaN        NaN        NaN   

  Unnamed: 6 Unnamed: 7 Unnamed: 8 Unnamed: 9  ... Unnamed: 67 Unnamed: 68  \
0        NaN        NaN        NaN        NaN  ...         NaN         NaN   
1        NaN        NaN        NaN        NaN  ...         NaN         NaN   
2        NaN        NaN        NaN        NaN  ...         NaN         NaN   

  Unnamed: 69 Unnamed: 70 Unnamed: 71 Unnamed: 72 Unnamed: 73 Unnamed: 74  \
0         NaN         NaN         NaN         NaN         NaN         NaN   
1         NaN  

In [4]:
# Step 1: Remove empty columns
print("🧹 Cleaning step 1: Removing empty columns...")
df_clean = df.dropna(axis=1, how='all')  # Remove completely empty columns
df_clean = df_clean.loc[:, ~df_clean.columns.str.contains('^Unnamed')]  # Remove unnamed cols

print(f"Shape after removing empty columns: {df_clean.shape}")

# Step 2: Identify and clean column names
print(f"\n📋 Columns after cleanup: {df_clean.columns.tolist()}")

# Rename columns to meaningful names based on content
new_cols = {}
for i, col in enumerate(df_clean.columns):
    if 'fid' in str(col).lower():
        new_cols[col] = 'location'
    elif 'originalLanguage' in str(col):
        new_cols[col] = 'reviewer_id'
    elif i == 2:  # Third column (index 2) seems to be content/rating
        new_cols[col] = 'content'
    else:
        new_cols[col] = f'col_{i}'

df_clean.rename(columns=new_cols, inplace=True)
print(f"✅ Columns renamed: {df_clean.columns.tolist()}")

# Step 3: Keep only meaningful columns
keep_cols = ['location', 'reviewer_id', 'content']
df_clean = df_clean[[col for col in keep_cols if col in df_clean.columns]]

print(f"\n✅ Final shape: {df_clean.shape}")
print(f"Sample data:\n{df_clean.head(3)}")

🧹 Cleaning step 1: Removing empty columns...
Shape after removing empty columns: (1615, 3)

📋 Columns after cleanup: ['fid', 'originalLanguage', 'reviewerId']
✅ Columns renamed: ['location', 'reviewer_id', 'content']

✅ Final shape: (1615, 3)
Sample data:
  location     reviewer_id      content
0    Blida        علي زبير  استقبال جيد
1    Blida   LAMINE KEBBAB          3.5
2    Blida  Kamel Mederbel          3.5


In [5]:
# Step 4: Clean text content
print("\n🧹 Cleaning step 2: Text normalization...")

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text).strip()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special chars but keep arabic/french chars
    text = text.replace(';', ',')
    return text

df_clean['content'] = df_clean['content'].apply(clean_text)
df_clean['reviewer_id'] = df_clean['reviewer_id'].apply(clean_text)

# Step 5: Remove rows with empty content
df_clean = df_clean[df_clean['content'].str.len() > 0].reset_index(drop=True)

# Step 6: Remove duplicates
initial_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['content'], keep='first').reset_index(drop=True)

print(f"✅ After removing duplicates: {len(df_clean)} rows (removed {initial_len - len(df_clean)})")

# Step 7: Handle missing values
print(f"\n📊 Missing values:")
print(df_clean.isnull().sum())

print(f"\n✅ Cleaning complete!")
print(f"Final dataset shape: {df_clean.shape}")
print(f"\nSample cleaned data:")
print(df_clean.head(3))


🧹 Cleaning step 2: Text normalization...
✅ After removing duplicates: 204 rows (removed 1119)

📊 Missing values:
location       0
reviewer_id    0
content        0
dtype: int64

✅ Cleaning complete!
Final dataset shape: (204, 3)

Sample cleaned data:
  location    reviewer_id                                            content
0    Blida       علي زبير                                        استقبال جيد
1    Blida  LAMINE KEBBAB                                                3.5
2    Blida      Midou Sou  J'ai passé chez un jeune homme, il est très se...


In [6]:
# Step 8: Save cleaned data
output_path = r'C:\Users\Imadeddine\OneDrive\Documents\GitHub\Air_Algerie_analysis\Data\google_maps_reviews_cleaned.csv'

df_clean.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Cleaned data saved to:")
print(f"   {output_path}")
print(f"\n📊 Final Statistics:")
print(f"   Total rows: {len(df_clean)}")
print(f"   Locations: {df_clean['location'].nunique()}")
print(f"   Reviewers: {df_clean['reviewer_id'].nunique()}")
print(f"   Avg content length: {df_clean['content'].str.len().mean():.0f} characters")

print("\n" + "="*70)
print("✨ CLEANING COMPLETE! Data ready for analysis")
print("="*70)

✅ Cleaned data saved to:
   C:\Users\Imadeddine\OneDrive\Documents\GitHub\Air_Algerie_analysis\Data\google_maps_reviews_cleaned.csv

📊 Final Statistics:
   Total rows: 204
   Locations: 29
   Reviewers: 127
   Avg content length: 56 characters

✨ CLEANING COMPLETE! Data ready for analysis
